# От Score товара к выбору итоговой ассортиментной матрицы

Продолжение [`../Кластеризация точек продаж`](../Кластеризация%20точек%20продаж) - там точки были
кластеризованы, здесь для готовых кластеров строится и выбирается конкретная товарная матрица.

Три шага: (1) посчитать Score каждого товара внутри его юнита (группы взаимозаменяемых товаров), (2)
разделить юниты на классы A/B/C по вкладу в Score кластера, (3) сгенерировать несколько кандидатов на
итоговую матрицу разными методами и выбрать лучший, сравнив каждого с фактической (текущей) матрицей по
деньгам.

Подключение к БД здесь - функция-заглушка, а данные ниже - синтетический пример на нескольких товарах и
юнитах, не выгрузка из реальной системы.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import expon

pd.options.display.max_columns = 100

## Настройка параметров

In [ ]:
# Границы накопительной доли для ABC-классификации юнитов внутри кластера
ABC_LOWER_BOUND = 0.5   # до этой накопительной доли Score - класс A
ABC_UPPER_BOUND = 0.9   # до этой - класс B, дальше - класс C

# Несколько значений λ для экспоненциального метода отбора (чем больше λ, тем строже отбор)
LAMBDAS = [0.5, 1, 2]

# Несколько кандидатов ABC-порогов: для каждого класса юнита (A/B/C) - минимальный нормированный скор товара
# (доля товара внутри своего юнита), плюс общий минимальный скор юнита (score_min). Обычно такой список
# готовится вручную по бизнес-смыслу.
ABC_THRESHOLD_CANDIDATES = pd.DataFrame([
    {"method": "мягкий", "a": 0.15, "b": 0.20, "c": 0.25, "s_u": 0.0},
    {"method": "средний", "a": 0.25, "b": 0.30, "c": 0.35, "s_u": 0.0},
    {"method": "строгий", "a": 0.35, "b": 0.40, "c": 0.45, "s_u": 0.0},
])

## Функции

In [ ]:
def get_db_engine():
    """Заглушка вместо реального подключения к корпоративной БД."""
    raise NotImplementedError("Подключение к реальной БД в портфолио-версии не используется")

In [ ]:
def classify_units_abc(units: pd.DataFrame, lower_bound: float, upper_bound: float) -> pd.DataFrame:
    """
    Делит юниты на классы A/B/C по накопительной доле их Score внутри кластера: сортируем юниты кластера
    по убыванию доли, берём накопительную сумму - первые юниты, дающие вместе до `lower_bound` Score
    кластера, это класс A, до `upper_bound` - класс B, остальное - класс C.
    """
    units = units.sort_values(["clst", "share_by_cluster"], ascending=[True, False]).copy()
    units["cum_share"] = units.groupby("clst")["share_by_cluster"].cumsum()

    def status(cum_share: float) -> str:
        if cum_share <= lower_bound:
            return "A"
        elif cum_share <= upper_bound:
            return "B"
        return "C"

    units["Status"] = units["cum_share"].apply(status)
    return units

In [ ]:
def matrix_sign_by_abc_threshold(row, a_bound: float, b_bound: float, c_bound: float, score_min: float) -> int:
    """Входит ли товар в матрицу при заданной комбинации ABC-порогов - разный порог по нормированному скору
    товара в зависимости от класса его юнита, плюс минимальный скор юнита для класса C."""
    if row["Status"] == "A" and row["score_norm_ap"] >= a_bound:
        return 1
    elif row["Status"] == "B" and row["score_norm_ap"] >= b_bound:
        return 1
    elif row["Status"] == "C" and row["score_norm_ap"] >= c_bound and row["score_unit"] > score_min:
        return 1
    return 0


def matrix_sign_by_exponential_bound(row, lam: float) -> int:
    """
    Вероятностная альтернатива ABC-порогам: граница по CDF экспоненциального распределения от доли юнита
    в скоре кластера (`share_by_cluster`) - чем весомее юнит для кластера, тем ниже требуемый порог по
    товару внутри него (более "весомым" юнитам разрешена более широкая матрица). λ регулирует строгость.
    """
    bound = expon.cdf(1 - row["share_by_cluster"], scale=1 / lam)
    return 1 if row["score_norm_ap"] > bound else 0

In [ ]:
def evaluate_matrix_variant(df: pd.DataFrame, in_matrix_col: str, label) -> pd.DataFrame:
    """Сводка по одному варианту матрицы: сколько денег в остатке/выручке/прибыли и сколько позиций
    остаётся, если считать матрицей только строки с `in_matrix_col == 1`."""
    included = df[df[in_matrix_col] == 1]
    return pd.DataFrame([{
        "вариант": label,
        "stock_rub": included["stock_rub"].sum(),
        "revenue": included["revenue"].sum(),
        "profit": included["profit"].sum(),
        "позиций": len(included),
    }])

## Алгоритм выполнения

### Синтетические данные: товары внутри юнитов одного кластера

Ниже - не выгрузка, а придуманные числа для одного кластера: 12 товаров, поровну поделённые на 4 юнита,
у каждого свой скор и свои показатели по деньгам. На них дальше проверяется вся логика ноутбука - реальные
входные файлы (справочники юнитов, скоров, фактической матрицы) в портфолио не включены.

In [ ]:
np.random.seed(0)
n = 12
products = pd.DataFrame({
    "apID": [f"AP_{i:02d}" for i in range(n)],
    "clst": ["Кластер_1"] * n,
    "Unit": np.repeat([f"Юнит_{j}" for j in range(4)], 3),
    "score": np.round(np.random.uniform(0.1, 5, n), 2),
    "stock_rub": np.round(np.random.uniform(1000, 20000, n)),
    "revenue": np.round(np.random.uniform(500, 15000, n)),
    "profit": np.round(np.random.uniform(-500, 5000, n)),
})
products

### Score юнита и ABC-классификация

In [ ]:
units = products.groupby(["clst", "Unit"], as_index=False)["score"].sum().rename(columns={"score": "score_unit"})
units["score_total_cluster"] = units.groupby("clst")["score_unit"].transform("sum")
units["share_by_cluster"] = units["score_unit"] / units["score_total_cluster"]

units = classify_units_abc(units, ABC_LOWER_BOUND, ABC_UPPER_BOUND)
units[["Unit", "score_unit", "share_by_cluster", "cum_share", "Status"]]

In [ ]:
# нормированный скор товара внутри своего юнита (доля товара в скоре юнита) - используется порогами матрицы
products = products.merge(units[["clst", "Unit", "score_unit", "share_by_cluster", "Status"]], on=["clst", "Unit"])
products["score_norm_ap"] = products["score"] / products["score_unit"]
products.head()

### Кандидаты на матрицу: ABC-пороги

In [ ]:
results = []
for _, row in ABC_THRESHOLD_CANDIDATES.iterrows():
    col = f"in_matrix_{row['method']}"
    products[col] = products.apply(
        lambda r: matrix_sign_by_abc_threshold(r, row["a"], row["b"], row["c"], row["s_u"]), axis=1
    )
    results.append(evaluate_matrix_variant(products, col, row["method"]))

abc_variants = pd.concat(results, ignore_index=True)
abc_variants

### Кандидаты на матрицу: экспоненциальный порог

In [ ]:
results = []
for lam in LAMBDAS:
    col = f"in_matrix_exp_{lam}"
    products[col] = products.apply(lambda r: matrix_sign_by_exponential_bound(r, lam), axis=1)
    results.append(evaluate_matrix_variant(products, col, f"экспоненциальный (λ={lam})"))

exp_variants = pd.concat(results, ignore_index=True)
exp_variants

### Сравнение всех кандидатов с фактической матрицей

In [ ]:
# для примера считаем фактической матрицей товары с товаром выше медианного скора - в реальном расчёте
# фактическая матрица берётся из отдельного справочника (какие AP сейчас реально числятся в матрице точки)
products["in_matrix_fact"] = (products["score"] >= products["score"].median()).astype(int)
fact_variant = evaluate_matrix_variant(products, "in_matrix_fact", "факт")

all_variants = pd.concat([fact_variant, abc_variants, exp_variants], ignore_index=True)

fact_row = all_variants[all_variants["вариант"] == "факт"].iloc[0]
for metric in ["stock_rub", "revenue", "profit"]:
    all_variants[f"diff_{metric}"] = all_variants[metric] - fact_row[metric]

all_variants

Дальше выбор одного варианта - вручную, по этой таблице (в оригинальном расчёте - ещё и по
scatter-графику разниц): смотрим не на один показатель, а на соотношение изменений - интересны варианты,
где рост выручки/прибыли ощутимо больше, чем рост занятого в остатках капитала (или где сокращение остатка
не тянет вниз выручку/прибыль пропорционально сильнее). На этом синтетическом примере лучше всего выглядит
`экспоненциальный (λ=0.5)` - небольшой рост остатка (+2 655), но заметно больший рост выручки (+14 198) и
прибыли (+3 250); `строгий` же сокращает остаток, но выручка и прибыль падают даже сильнее пропорционально -
такой вариант, скорее всего, будет отклонён.

In [ ]:
selected = "exp_0.5"
final_matrix = products[products["in_matrix_exp_0.5"] == 1]
final_matrix